# LeetCode 1695 — Maximum Erasure Value bằng Python

Link đề: https://leetcode.com/problems/maximum-erasure-value/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, Sliding Window

Skeleton LeetCode:

```python
class Solution:
    def maximumUniqueSubarray(self, nums: List[int]) -> int:
        ...
```

Lộ trình: **vét cạn mọi đoạn con** (`O(n²)`/`O(n³)`) → **cửa sổ trượt + set** (`O(n)`) → **cửa sổ "nhảy" bằng vị trí xuất hiện cuối + tổng tiền tố** → **mảng đếm** thay cho set vì `nums[i] ≤ 10⁴`. Đây là bài anh em của *Longest Substring Without Repeating Characters* (LeetCode 3) — chỉ đổi "độ dài" thành "tổng".

## 1. Mục tiêu học

1. Hiểu định nghĩa **đoạn con (subarray)** liên tiếp và điều kiện "mọi phần tử phân biệt".
2. Nắm kỹ thuật **cửa sổ trượt hai con trỏ** `l, r`: mở rộng phải, co trái khi vi phạm, bảo toàn invariant "cửa sổ không có phần tử trùng".
3. Dùng **set / mảng đếm / từ điển vị trí cuối** để kiểm tra trùng trong `O(1)`.
4. Kết hợp **tổng tiền tố** để tính tổng đoạn khi con trỏ trái **nhảy** một bước lớn.
5. Lý giải độ phức tạp **khấu hao (amortized)** `O(n)`: mỗi phần tử vào và ra cửa sổ tối đa một lần.

## 2. Đề bài & ví dụ

Cho mảng số nguyên **dương** `nums`. Bạn muốn xóa **một đoạn con** gồm các phần tử **đôi một khác nhau**. Điểm nhận được bằng **tổng** các phần tử của đoạn đó. Trả về điểm lớn nhất có thể khi xóa đúng một đoạn con.

Đoạn con `b` của `a` là dãy liên tiếp `a[l], a[l+1], …, a[r]`.

**Ví dụ 1:** `nums = [4,2,4,5,6]` → `17`. Đoạn tối ưu `[2,4,5,6]`.

**Ví dụ 2:** `nums = [5,2,1,2,5,2,1,2,5]` → `8`. Đoạn tối ưu `[5,2,1]` hoặc `[1,2,5]`.

**Ràng buộc:**
- `1 ≤ nums.length ≤ 10⁵`
- `1 ≤ nums[i] ≤ 10⁴`

In [1]:
import random
import time
from itertools import accumulate
from typing import List

examples = [
    ([4, 2, 4, 5, 6], 17),
    ([5, 2, 1, 2, 5, 2, 1, 2, 5], 8),
]
for nums, want in examples:
    print(f"nums={nums} -> kỳ vọng {want}")

nums=[4, 2, 4, 5, 6] -> kỳ vọng 17
nums=[5, 2, 1, 2, 5, 2, 1, 2, 5] -> kỳ vọng 8


## 3. Kiến thức nền

### 3.1 Đoạn con và slicing

`nums[l:r+1]` là đoạn con từ `l` đến `r`. Có `n(n+1)/2` đoạn con khác rỗng. Kiểm tra "phân biệt" nhanh: `len(set(seg)) == len(seg)`.

In [2]:
nums = [4, 2, 4, 5, 6]
for l in range(len(nums)):
    for r in range(l, len(nums)):
        seg = nums[l:r + 1]
        if len(set(seg)) == len(seg) and len(seg) >= 3:
            print(f"[{l},{r}] {seg} tổng={sum(seg)}")

[1,3] [2, 4, 5] tổng=11
[1,4] [2, 4, 5, 6] tổng=17
[2,4] [4, 5, 6] tổng=15


### 3.2 `set` — thêm, xóa, kiểm tra thành viên trung bình `O(1)`

In [3]:
seen = set()
for x in [4, 2, 4]:
    print(f"{x} đã có? {x in seen}")
    seen.add(x)
seen.discard(4)
print("sau khi bỏ 4:", seen)

4 đã có? False
2 đã có? False
4 đã có? True
sau khi bỏ 4: {2}


### 3.3 Tổng tiền tố

`pre[k] = nums[0] + … + nums[k−1]`. Tổng đoạn `[l, r]` = `pre[r+1] − pre[l]`. Hữu ích khi con trỏ trái nhảy xa (không phải trừ từng phần tử).

In [4]:
pre = [0] + list(accumulate([4, 2, 4, 5, 6]))
print("pre =", pre, "| tổng [1,4] =", pre[5] - pre[1])

pre = [0, 4, 6, 10, 15, 21] | tổng [1,4] = 17


### 3.4 Mảng đếm thay cho set

Khi giá trị nằm trong miền nhỏ (`1..10⁴`), mảng `cnt = [0] * 10001` nhanh hơn set (không băm). `cnt[x] > 0` ⇔ `x` đang trong cửa sổ.

In [5]:
cnt = [0] * 11
for x in [4, 2, 4]:
    cnt[x] += 1
print("cnt[4] =", cnt[4], "-> 4 xuất hiện 2 lần, cửa sổ không hợp lệ")

cnt[4] = 2 -> 4 xuất hiện 2 lần, cửa sổ không hợp lệ


### 3.5 Khuôn cửa sổ trượt "co khi vi phạm"

```text
l = 0
for r in range(n):
    thêm a[r] vào cửa sổ
    while cửa sổ vi phạm:
        bỏ a[l]; l += 1
    cập nhật đáp án với cửa sổ [l, r]
```

## 4. Ý tưởng nền tảng / trực giác

Làm tay ví dụ 1 `[4,2,4,5,6]`, mỗi khi gặp phần tử đã có trong đoạn hiện tại thì "cắt" đoạn ngay sau lần xuất hiện cũ:

| r | nums[r] | đoạn hợp lệ dài nhất kết thúc tại r | tổng |
|---|---|---|---|
| 0 | 4 | [4] | 4 |
| 1 | 2 | [4,2] | 6 |
| 2 | 4 | 4 trùng ⇒ cắt sau 4 cũ: [2,4] | 6 |
| 3 | 5 | [2,4,5] | 11 |
| 4 | 6 | [2,4,5,6] | **17** |

**Nhận xét:** vì mọi số đều **dương**, với mỗi `r` cố định, đoạn hợp lệ **dài nhất** kết thúc tại `r` cũng có **tổng lớn nhất**. Nên chỉ cần, với mỗi `r`, tìm `l` nhỏ nhất hợp lệ.

In [6]:
nums = [4, 2, 4, 5, 6]
for r in range(len(nums)):
    l = r
    while l - 1 >= 0 and nums[l - 1] not in nums[l:r + 1]:
        l -= 1
    print(f"r={r}: l nhỏ nhất = {l}, đoạn {nums[l:r + 1]}, tổng {sum(nums[l:r + 1])}")

r=0: l nhỏ nhất = 0, đoạn [4], tổng 4
r=1: l nhỏ nhất = 0, đoạn [4, 2], tổng 6
r=2: l nhỏ nhất = 1, đoạn [2, 4], tổng 6
r=3: l nhỏ nhất = 1, đoạn [2, 4, 5], tổng 11
r=4: l nhỏ nhất = 1, đoạn [2, 4, 5, 6], tổng 17


## 5. Cách 1 — Brute Force / baseline: thử mọi điểm bắt đầu

**Ý tưởng:** với mỗi `l`, mở rộng `r` sang phải, dùng set để biết khi nào gặp trùng thì **dừng** (vì mở rộng thêm cũng vẫn trùng). Cộng dồn tổng trong lúc mở rộng.

In [7]:
def max_erasure_brute(nums):
    best = 0
    n = len(nums)
    for l in range(n):
        seen = set()
        s = 0
        for r in range(l, n):
            if nums[r] in seen:
                break
            seen.add(nums[r])
            s += nums[r]
            best = max(best, s)
    return best

for nums, want in examples:
    print(max_erasure_brute(nums), "kỳ vọng", want)

17 kỳ vọng 17
8 kỳ vọng 8


**Độ phức tạp:** worst case (mọi phần tử khác nhau) Time `O(n²)`, Space `O(n)`. Với `n = 10⁵` là `5·10⁹` phép — quá chậm.

**Nút thắt:** khi chuyển từ `l` sang `l+1`, ta **xây lại set từ đầu**, dù đoạn `[l+1, r−1]` vẫn hợp lệ. Cửa sổ trượt tái sử dụng công việc đó.

In [8]:
for n in [1000, 3000]:
    arr = list(range(1, n + 1))       # toàn phân biệt -> worst case
    t0 = time.perf_counter(); max_erasure_brute(arr)
    print(f"n={n}: {time.perf_counter() - t0:.3f}s")

n=1000: 0.054s


n=3000: 0.321s


## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (đơn điệu):** nếu `[l, r]` không hợp lệ (có trùng) thì mọi `[l', r]` với `l' < l` cũng không hợp lệ. Ngược lại nếu `[l, r]` hợp lệ thì mọi đoạn con của nó hợp lệ. Do đó khi `r` tăng, `l` tối ưu **không bao giờ giảm** ⇒ hai con trỏ cùng chạy về một phía.

**Insight 2 (số dương):** với `r` cố định, đoạn dài nhất ⇒ tổng lớn nhất. Không cần xét các `l` lớn hơn.

**Invariant:** sau khi xử lý `r`, cửa sổ `[l, r]` không có phần tử trùng và `l` nhỏ nhất có thể; `window_sum = sum(nums[l..r])`.

**Quy tắc:**
- Nếu `nums[r]` chưa có trong cửa sổ ⇒ thêm vào, cộng tổng.
- Nếu đã có ⇒ bỏ dần `nums[l]` (trừ tổng, `l += 1`) cho tới khi bản sao cũ của `nums[r]` bị bỏ ra.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Cửa sổ trượt + set

Từng đoạn:
1. `seen` = set các giá trị trong cửa sổ, `s` = tổng cửa sổ, `l = 0`.
2. Với mỗi `r`: trong khi `nums[r] in seen`, bỏ `nums[l]` khỏi `seen`, `s -= nums[l]`, `l += 1`.
3. Thêm `nums[r]`, `s += nums[r]`, cập nhật `best`.

In [9]:
def max_erasure_set(nums):
    seen = set()
    l = s = best = 0
    for r, x in enumerate(nums):
        while x in seen:
            seen.remove(nums[l])
            s -= nums[l]
            l += 1
        seen.add(x)
        s += x
        if s > best:
            best = s
    return best

for nums, want in examples:
    print(max_erasure_set(nums), "kỳ vọng", want)

17 kỳ vọng 17
8 kỳ vọng 8


Code nộp LeetCode của Cách 2:

```python
class Solution:
    def maximumUniqueSubarray(self, nums: List[int]) -> int:
        seen = set()
        l = s = best = 0
        for x in nums:
            while x in seen:
                seen.remove(nums[l])
                s -= nums[l]
                l += 1
            seen.add(x)
            s += x
            best = max(best, s)
        return best
```

**Độ phức tạp:** mỗi chỉ số được thêm vào `seen` đúng 1 lần và bỏ ra tối đa 1 lần ⇒ tổng số vòng `while` ≤ `n` ⇒ Time `O(n)` (khấu hao), Space `O(min(n, U))` với `U` là số giá trị khác nhau.

### 7.2 Cách 3 — Nhảy con trỏ trái bằng "vị trí xuất hiện cuối" + tổng tiền tố

Thay vì bỏ từng phần tử, ghi `last[x]` = chỉ số gần nhất của `x`. Khi gặp `x = nums[r]` đã xuất hiện ở vị trí `p = last[x] ≥ l`, đặt ngay `l = p + 1`. Tổng cửa sổ tính bằng `pre[r+1] − pre[l]`.

Lưu ý: `l = max(l, last[x] + 1)` — phải lấy `max` vì `last[x]` có thể nằm **ngoài** cửa sổ (trước `l`).

In [10]:
def max_erasure_jump(nums):
    pre = [0]
    for x in nums:
        pre.append(pre[-1] + x)
    last = {}
    l = best = 0
    for r, x in enumerate(nums):
        if x in last and last[x] + 1 > l:
            l = last[x] + 1
        last[x] = r
        cur = pre[r + 1] - pre[l]
        if cur > best:
            best = cur
    return best

for nums, want in examples:
    print(max_erasure_jump(nums), "kỳ vọng", want)

17 kỳ vọng 17
8 kỳ vọng 8


Code nộp LeetCode của Cách 3:

```python
class Solution:
    def maximumUniqueSubarray(self, nums: List[int]) -> int:
        pre = [0]
        for x in nums:
            pre.append(pre[-1] + x)
        last = {}
        l = best = 0
        for r, x in enumerate(nums):
            if x in last:
                l = max(l, last[x] + 1)
            last[x] = r
            best = max(best, pre[r + 1] - pre[l])
        return best
```

**Độ phức tạp:** Time `O(n)` (không khấu hao — mỗi bước `O(1)`), Space `O(n)` cho `pre` và `last`.

### 7.3 Cách 4 — Mảng vị trí cuối kích thước 10⁴ + 1

Giống Cách 3 nhưng `last` là list `[-1] * 10001` (giá trị ≤ 10⁴) — tránh băm, chạy nhanh hơn trong Python. Tổng cửa sổ cũng có thể duy trì bằng cách trừ dần nếu không muốn mảng `pre`; ở đây dùng `pre`.

In [11]:
def max_erasure_array(nums):
    last = [-1] * 10001
    pre = [0] * (len(nums) + 1)
    l = best = 0
    for r, x in enumerate(nums):
        pre[r + 1] = pre[r] + x
        p = last[x]
        if p >= l:
            l = p + 1
        last[x] = r
        cur = pre[r + 1] - pre[l]
        if cur > best:
            best = cur
    return best

for nums, want in examples:
    print(max_erasure_array(nums), "kỳ vọng", want)

17 kỳ vọng 17
8 kỳ vọng 8


**Độ phức tạp:** Time `O(n + V)` với `V = 10⁴` (khởi tạo mảng), Space `O(n + V)`.

## 8. TRACE từng bước

Trace Cách 2 trên ví dụ 2 `[5,2,1,2,5,2,1,2,5]`:

| r | x | bỏ ra (co trái) | cửa sổ sau | s | best |
|---|---|---|---|---|---|
| 0 | 5 | — | [5] | 5 | 5 |
| 1 | 2 | — | [5,2] | 7 | 7 |
| 2 | 1 | — | [5,2,1] | 8 | 8 |
| 3 | 2 | 5, 2 | [1,2] | 3 | 8 |
| 4 | 5 | — | [1,2,5] | 8 | 8 |
| 5 | 2 | 1, 2 | [5,2] | 7 | 8 |
| 6 | 1 | — | [5,2,1] | 8 | 8 |
| 7 | 2 | 5, 2 | [1,2] | 3 | 8 |
| 8 | 5 | — | [1,2,5] | 8 | 8 |

In [12]:
def trace_set(nums):
    seen = set(); l = s = best = 0
    for r, x in enumerate(nums):
        removed = []
        while x in seen:
            seen.remove(nums[l]); s -= nums[l]; removed.append(nums[l]); l += 1
        seen.add(x); s += x; best = max(best, s)
        print(f"r={r} x={x} bỏ={removed or '-'} cửa sổ=[{l},{r}] {nums[l:r + 1]} s={s} best={best}")
    return best

trace_set([5, 2, 1, 2, 5, 2, 1, 2, 5])

r=0 x=5 bỏ=- cửa sổ=[0,0] [5] s=5 best=5
r=1 x=2 bỏ=- cửa sổ=[0,1] [5, 2] s=7 best=7
r=2 x=1 bỏ=- cửa sổ=[0,2] [5, 2, 1] s=8 best=8
r=3 x=2 bỏ=[5, 2] cửa sổ=[2,3] [1, 2] s=3 best=8
r=4 x=5 bỏ=- cửa sổ=[2,4] [1, 2, 5] s=8 best=8
r=5 x=2 bỏ=[1, 2] cửa sổ=[4,5] [5, 2] s=7 best=8
r=6 x=1 bỏ=- cửa sổ=[4,6] [5, 2, 1] s=8 best=8
r=7 x=2 bỏ=[5, 2] cửa sổ=[6,7] [1, 2] s=3 best=8
r=8 x=5 bỏ=- cửa sổ=[6,8] [1, 2, 5] s=8 best=8


8

In [13]:
def trace_jump(nums):
    pre = [0] + list(accumulate(nums)); last = {}; l = best = 0
    for r, x in enumerate(nums):
        old_l = l
        if x in last:
            l = max(l, last[x] + 1)
        last[x] = r
        cur = pre[r + 1] - pre[l]; best = max(best, cur)
        jump = f"l: {old_l}->{l}" if l != old_l else f"l={l}"
        print(f"r={r} x={x} {jump:9s} last={dict(sorted(last.items()))} tổng={cur} best={best}")
    return best

trace_jump([4, 2, 4, 5, 6])

r=0 x=4 l=0       last={4: 0} tổng=4 best=4
r=1 x=2 l=0       last={2: 1, 4: 0} tổng=6 best=6
r=2 x=4 l: 0->1   last={2: 1, 4: 2} tổng=6 best=6
r=3 x=5 l=1       last={2: 1, 4: 2, 5: 3} tổng=11 best=11
r=4 x=6 l=1       last={2: 1, 4: 2, 5: 3, 6: 4} tổng=17 best=17


17

## 9. Code nộp LeetCode

In [14]:
class Solution:
    def maximumUniqueSubarray(self, nums: List[int]) -> int:
        seen = set()
        l = s = best = 0
        for x in nums:
            while x in seen:
                seen.remove(nums[l])
                s -= nums[l]
                l += 1
            seen.add(x)
            s += x
            if s > best:
                best = s
        return best

sol = Solution()
for nums, want in examples:
    got = sol.maximumUniqueSubarray(nums)
    print(got, "OK" if got == want else "SAI")

17 OK
8 OK


## 10. Vì sao thuật toán đúng

Gọi `L(r)` là chỉ số trái nhỏ nhất sao cho `nums[L(r)..r]` không có phần tử trùng.

**(a) `L(r)` không giảm.** Nếu `nums[L(r+1)..r+1]` không trùng thì đoạn con `nums[L(r+1)..r]` cũng không trùng, nên `L(r) ≤ L(r+1)`.

**(b) Cách 2 duy trì `l = L(r)`.** Giả sử trước bước `r`, `l = L(r−1)`. Khi thêm `x = nums[r]`: nếu `x` không có trong cửa sổ thì `[l, r]` hợp lệ và không thể mở rộng sang trái (vì `[l−1, r−1]` đã không hợp lệ) ⇒ `L(r) = l`. Nếu `x` có trong cửa sổ ở vị trí `p`, vòng `while` bỏ đúng các phần tử `l..p` — cửa sổ mới `[p+1, r]` hợp lệ, và mọi `l' ≤ p` đều chứa cả `nums[p]` lẫn `nums[r]` ⇒ `L(r) = p + 1`.

**(c) Đáp án.** Mọi đoạn hợp lệ `[l', r]` có `l' ≥ L(r)` nên là đoạn con của `[L(r), r]`; vì mọi số **dương**, `sum(nums[l'..r]) ≤ sum(nums[L(r)..r])`. Vậy `max_r sum(nums[L(r)..r])` là đáp án — đúng thứ Cách 2 tính.

Cách 3/4 tính trực tiếp `L(r) = max(L(r−1), last[nums[r]] + 1)`, cùng giá trị.

## 11. Phân tích Time / Space

Gọi `n = len(nums) ≤ 10⁵`, `V = 10⁴` là giá trị lớn nhất.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Vét cạn mọi `l` | `O(n²)` worst | `O(n)` | worst khi mọi phần tử khác nhau |
| 2. Cửa sổ + set | `O(n)` khấu hao | `O(min(n, V))` | mỗi phần tử vào/ra 1 lần |
| 3. Nhảy + dict + prefix | `O(n)` | `O(n)` | không có vòng `while` |
| 4. Nhảy + mảng | `O(n + V)` | `O(n + V)` | nhanh nhất trong Python |

In [15]:
random.seed(5)
big = [random.randint(1, 10**4) for _ in range(10**5)]
for name, fn in [("set", max_erasure_set), ("jump", max_erasure_jump), ("array", max_erasure_array)]:
    t0 = time.perf_counter(); r = fn(big)
    print(f"{name:5s}: {r}  {time.perf_counter() - t0:.3f}s")

set  : 2116465  0.013s
jump : 2116465  0.017s
array: 2116465  0.012s


## 12. Bộ test

In [16]:
fns = [max_erasure_brute, max_erasure_set, max_erasure_jump, max_erasure_array,
       lambda a: Solution().maximumUniqueSubarray(a)]

for nums, want in examples:
    for fn in fns:
        assert fn(nums) == want

edge = [
    ([7], 7),                         # một phần tử
    ([3, 3, 3, 3], 3),                # toàn trùng
    ([1, 2, 3, 4], 10),               # toàn khác -> cả mảng
    ([10000, 1, 10000], 10001),
    ([1, 2, 1, 3, 4, 3], 10),         # [2,1,3,4] = 10
    ([5, 1, 2, 5], 8),                # cửa sổ [1,2,5] hoặc [5,1,2]
]
for nums, want in edge:
    assert max_erasure_brute(nums) == want, nums
    for fn in fns:
        assert fn(nums) == want, (fn, nums)

random.seed(2024)
for _ in range(1500):
    n = random.randint(1, 60)
    hi = random.choice([3, 10, 50, 10**4])
    nums = [random.randint(1, hi) for _ in range(n)]
    want = max_erasure_brute(nums)
    for fn in fns[1:]:
        assert fn(nums) == want, nums
print("ví dụ + edge + 1500 bộ ngẫu nhiên: OK")

ví dụ + edge + 1500 bộ ngẫu nhiên: OK


In [17]:
# kiểm tra thêm: brute force O(n^3) thuần định nghĩa (không dùng break) trên mảng rất nhỏ
def max_erasure_def(nums):
    n = len(nums)
    return max(sum(nums[l:r + 1]) for l in range(n) for r in range(l, n)
               if len(set(nums[l:r + 1])) == r - l + 1)

random.seed(1)
for _ in range(500):
    nums = [random.randint(1, 5) for _ in range(random.randint(1, 10))]
    assert max_erasure_def(nums) == max_erasure_set(nums)
print("khớp định nghĩa gốc trên 500 mảng nhỏ")

khớp định nghĩa gốc trên 500 mảng nhỏ


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Cách 3 quên `max` khi nhảy con trỏ trái

Viết `l = last[x] + 1` khiến `l` **lùi lại** nếu lần xuất hiện cũ của `x` đã nằm ngoài cửa sổ.

In [18]:
def wrong_no_max(nums):
    pre = [0] + list(accumulate(nums)); last = {}; l = best = 0
    for r, x in enumerate(nums):
        if x in last:
            l = last[x] + 1            # SAI: có thể lùi
        last[x] = r
        best = max(best, pre[r + 1] - pre[l])
    return best

nums = [1, 2, 3, 2, 1]
print("sai =", wrong_no_max(nums), "| đúng =", max_erasure_set(nums))

sai = 8 | đúng = 6


Với `[1,2,3,2,1]`: tại `r = 3` (số 2), `l` nhảy tới 2. Tại `r = 4` (số 1), `last[1] = 0` ⇒ `l` bị đặt lại về 1 — cửa sổ `[2,3,2,1]` chứa hai số 2! **Sửa:** `l = max(l, last[x] + 1)`.

### Lỗi 2 — Dùng `if` thay cho `while` khi co cửa sổ

Chỉ bỏ **một** phần tử bên trái, trong khi bản sao của `x` có thể nằm sâu hơn.

In [19]:
def wrong_if(nums):
    seen = set(); l = s = best = 0
    for x in nums:
        if x in seen:                  # SAI: phải là while
            seen.remove(nums[l]); s -= nums[l]; l += 1
        seen.add(x); s += x; best = max(best, s)
    return best

nums = [4, 2, 4, 5, 6]
print("sai =", wrong_if(nums), "| đúng =", max_erasure_set(nums))

sai = 17 | đúng = 17


Ở ví dụ 1 bản sao của `4` nằm ngay đầu cửa sổ nên bỏ một phần tử là đủ — lỗi bị che giấu. Thử `[2,1,3,1,5]`: tại `r = 3` (số 1), bản sao của 1 ở vị trí 1, cần bỏ cả `2` và `1`, nhưng `if` chỉ bỏ `2` ⇒ cửa sổ thực `[1,3,1]` có trùng, và tổng bị tính sai.

In [20]:
nums = [2, 1, 3, 1, 5]
print("sai =", wrong_if(nums), "| đúng =", max_erasure_set(nums))

sai = 10 | đúng = 9


**Sửa:** `while x in seen:`.

### Lỗi 3 — Đi tìm "đoạn dài nhất" rồi trả về tổng của nó

Hai đoạn hợp lệ có cùng/khác độ dài có thể có tổng khác nhau; đoạn dài nhất toàn cục **không nhất thiết** có tổng lớn nhất.

In [21]:
def wrong_longest(nums):
    seen = set(); l = 0; best_len = 0; best_sum = 0
    for r, x in enumerate(nums):
        while x in seen:
            seen.remove(nums[l]); l += 1
        seen.add(x)
        if r - l + 1 > best_len:
            best_len = r - l + 1; best_sum = sum(nums[l:r + 1])
    return best_sum

random.seed(4)
for _ in range(5000):
    nums = [random.randint(1, 9) for _ in range(random.randint(2, 9))]
    if wrong_longest(nums) != max_erasure_set(nums):
        print("phản ví dụ:", nums, "| sai =", wrong_longest(nums), "| đúng =", max_erasure_set(nums))
        break

phản ví dụ: [1, 4, 9, 9, 6, 5] | sai = 14 | đúng = 20


**Sửa:** so sánh theo **tổng** ở mọi `r` (với `r` cố định thì dài nhất ⇔ tổng lớn nhất, nhưng giữa các `r` khác nhau thì không).

### Lỗi 4 — Tính tổng cửa sổ bằng `sum(nums[l:r+1])` mỗi bước

Đúng nhưng `O(n)` mỗi bước ⇒ tổng `O(n²)` worst case, TLE.

In [22]:
def slow_sum(nums):
    seen = set(); l = best = 0
    for r, x in enumerate(nums):
        while x in seen:
            seen.remove(nums[l]); l += 1
        seen.add(x); best = max(best, sum(nums[l:r + 1]))
    return best

arr = list(range(1, 6001))
t0 = time.perf_counter(); slow_sum(arr); t1 = time.perf_counter(); max_erasure_set(arr); t2 = time.perf_counter()
print(f"slow_sum: {t1 - t0:.3f}s | cửa sổ duy trì tổng: {t2 - t1:.4f}s (n = 6000)")

slow_sum: 0.110s | cửa sổ duy trì tổng: 0.0004s (n = 6000)


**Sửa:** duy trì biến `s` (cộng khi thêm, trừ khi bỏ) hoặc dùng tổng tiền tố.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Đoạn con **liên tiếp** + điều kiện **đơn điệu** (đúng với đoạn lớn ⇒ đúng với đoạn con)" ⇒ **cửa sổ trượt**.
- "Không lặp lại" ⇒ set / mảng đếm / `last[x]`.
- Mảng **dương** + tối đa tổng ⇒ cửa sổ dài nhất tại mỗi `r` là tốt nhất. Nếu có số âm, lập luận này hỏng (cần kỹ thuật khác).
- Bài sinh đôi: **LeetCode 3** (độ dài chuỗi con không lặp ký tự) — cùng code, đổi `s` thành `r − l + 1`.
- Khi giá trị ≤ 10⁴–10⁶, mảng thay dict giúp tăng tốc 2–3 lần trong Python.

## 15. Biến thể của bài

1. **LeetCode 3 — Longest Substring Without Repeating Characters:** trả về độ dài thay vì tổng.
2. **Mỗi giá trị được xuất hiện tối đa `k` lần:** dùng mảng đếm, co khi `cnt[x] > k`.
3. **Trả về chính đoạn con** (chỉ số `l, r`): lưu `l, r` khi cập nhật `best`.
4. **Có số âm:** cửa sổ dài nhất không còn chắc tổng lớn nhất; với mỗi `r` phải chọn `l ∈ [L(r), r]` tối thiểu `pre[l]` ⇒ cửa sổ + **hàng đợi đơn điệu** trên `pre` (min-queue).
5. **Đếm số đoạn con phân biệt:** cộng `r − L(r) + 1` cho mọi `r`.

In [23]:
def max_sum_at_most_k(nums, k):
    """Biến thể 2: mỗi giá trị xuất hiện tối đa k lần trong đoạn."""
    from collections import Counter
    cnt = Counter(); l = s = best = 0
    for x in nums:
        cnt[x] += 1; s += x
        while cnt[x] > k:
            cnt[nums[l]] -= 1; s -= nums[l]; l += 1
        best = max(best, s)
    return best

def count_distinct_subarrays(nums):
    """Biến thể 5: số đoạn con có các phần tử phân biệt."""
    last = {}; l = total = 0
    for r, x in enumerate(nums):
        if x in last:
            l = max(l, last[x] + 1)
        last[x] = r; total += r - l + 1
    return total

print("k=1 trùng bài gốc:", max_sum_at_most_k([4, 2, 4, 5, 6], 1), "| k=2:", max_sum_at_most_k([4, 2, 4, 5, 6], 2))
print("số đoạn con phân biệt của [1,2,1]:", count_distinct_subarrays([1, 2, 1]), "( [1],[2],[1],[1,2],[2,1] )")

k=1 trùng bài gốc: 17 | k=2: 21
số đoạn con phân biệt của [1,2,1]: 5 ( [1],[2],[1],[1,2],[2,1] )


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | mọi `l`, mở rộng tới khi trùng | `O(n²)` | `O(n)` | kiểm chứng |
| 2. Cửa sổ + set | co từng bước | `O(n)` khấu hao | `O(n)` | **nộp bài**, dễ nhớ |
| 3. Nhảy + `last` dict + prefix | co một phát | `O(n)` | `O(n)` | khi cần chỉ số chính xác |
| 4. Nhảy + mảng | như 3, không băm | `O(n + V)` | `O(n + V)` | tối ưu hằng số |

## 17. Bài tập (từ dễ đến khó)

**Bài 1 (đoán output):** `maximumUniqueSubarray([1, 1, 2, 2, 3, 3])` bằng bao nhiêu?

**Bài 2 (tính tay):** trace Cách 3 trên `[3, 1, 3, 2, 1]`: ghi `l` và tổng ở mỗi `r`.

**Bài 3 (điền TODO):** hoàn thiện `best_segment(nums)` trả về cặp `(l, r)` của đoạn tối ưu (chọn đoạn có `l` nhỏ nhất nếu hòa).

**Bài 4 (tự viết lại):** viết lại Cách 2 nhưng dùng **mảng đếm** `cnt` thay cho set.

**Bài 5 (mini challenge):** cho phép `nums[i]` **âm** (`-10⁴ ≤ nums[i] ≤ 10⁴`), vẫn yêu cầu đoạn **khác rỗng** có phần tử phân biệt và tổng lớn nhất. Viết lời giải `O(n)` hoặc `O(n log n)` và kiểm chứng bằng vét cạn. (Biến thể tự đặt.)

In [24]:
def best_segment(nums):
    # TODO: cửa sổ trượt; khi s > best thì lưu (l, r)
    pass

## 18. Gợi ý 3 mức

**Bài 1:** (1) Đoạn hợp lệ không chứa hai số bằng nhau liền kề. (2) Đoạn tốt nhất có dạng `[1,2]`, `[2,3]`. (3) Đáp án `5`.

**Bài 2:** (1) `r=0: l=0, 3`. (2) `r=2`: gặp 3 ở vị trí 0 ⇒ `l=1`. (3) `r=4`: gặp 1 ở vị trí 1 ⇒ `l=2`, tổng `3+2+1 = 6`.

**Bài 3:** (1) Dùng Cách 2. (2) Lưu `l` hiện tại và `r` khi `s > best` (dùng `>` để giữ đoạn đầu tiên). (3) Trả về tuple.

**Bài 4:** (1) `cnt = [0]*10001`. (2) Thêm: `cnt[x] += 1`. (3) `while cnt[x] > 1: cnt[nums[l]] -= 1; ...`.

**Bài 5:** (1) Với `r` cố định, `l` chỉ được trong `[L(r), r]`; tổng = `pre[r+1] − pre[l]`. (2) Cần `min pre[l]` trên cửa sổ trượt `[L(r), r]` — hai đầu đều chỉ tiến ⇒ **deque đơn điệu**. (3) Đẩy `l = r` vào deque trước khi truy vấn để bảo đảm đoạn khác rỗng.

## 19. Đáp án tham khảo

In [25]:
from collections import deque

# Bài 1
assert Solution().maximumUniqueSubarray([1, 1, 2, 2, 3, 3]) == 5

# Bài 2
assert max_erasure_jump([3, 1, 3, 2, 1]) == 6

# Bài 3
def best_segment_ans(nums):
    seen = set(); l = s = 0; best = -1; seg = None
    for r, x in enumerate(nums):
        while x in seen:
            seen.remove(nums[l]); s -= nums[l]; l += 1
        seen.add(x); s += x
        if s > best:
            best, seg = s, (l, r)
    return seg
assert best_segment_ans([4, 2, 4, 5, 6]) == (1, 4)
assert best_segment_ans([5, 2, 1, 2, 5, 2, 1, 2, 5]) == (0, 2)

# Bài 4
def max_erasure_cnt(nums):
    cnt = [0] * 10001; l = s = best = 0
    for x in nums:
        cnt[x] += 1; s += x
        while cnt[x] > 1:
            cnt[nums[l]] -= 1; s -= nums[l]; l += 1
        best = max(best, s)
    return best

random.seed(12)
for _ in range(500):
    nums = [random.randint(1, 20) for _ in range(random.randint(1, 50))]
    assert max_erasure_cnt(nums) == max_erasure_brute(nums)
print("Bài 1-4: OK")

Bài 1-4: OK


In [26]:
# Bài 5 — có số âm: cửa sổ L(r) + deque đơn điệu tìm min pre[l], l in [L(r), r]
def max_unique_with_neg(nums):
    n = len(nums)
    pre = [0] * (n + 1)
    for i, x in enumerate(nums):
        pre[i + 1] = pre[i] + x
    last = {}; L = 0; dq = deque(); best = None
    for r, x in enumerate(nums):
        if x in last:
            L = max(L, last[x] + 1)
        last[x] = r
        while dq and pre[dq[-1]] >= pre[r]:   # thêm ứng viên l = r
            dq.pop()
        dq.append(r)
        while dq[0] < L:                      # loại l ngoài cửa sổ
            dq.popleft()
        cur = pre[r + 1] - pre[dq[0]]
        best = cur if best is None else max(best, cur)
    return best

def brute_with_neg(nums):
    n = len(nums); best = None
    for l in range(n):
        seen = set(); s = 0
        for r in range(l, n):
            if nums[r] in seen: break
            seen.add(nums[r]); s += nums[r]
            best = s if best is None else max(best, s)
    return best

random.seed(13)
for _ in range(1000):
    nums = [random.randint(-6, 6) for _ in range(random.randint(1, 25))]
    assert max_unique_with_neg(nums) == brute_with_neg(nums), nums
print("Bài 5: khớp vét cạn trên 1000 mảng có số âm; ví dụ [3,-10,5,3,2] ->", max_unique_with_neg([3, -10, 5, 3, 2]))

Bài 5: khớp vét cạn trên 1000 mảng có số âm; ví dụ [3,-10,5,3,2] -> 10


## 20. Template nhớ nhanh + Checklist

```python
# Cửa sổ trượt "không lặp phần tử", tối đa tổng (mảng dương)
seen = set(); l = s = best = 0
for x in nums:
    while x in seen:              # co trái cho đến khi hết trùng
        seen.remove(nums[l]); s -= nums[l]; l += 1
    seen.add(x); s += x
    best = max(best, s)
return best
```

**Checklist trước khi submit:**
- [ ] Dùng `while` (không phải `if`) khi co cửa sổ.
- [ ] Bản "nhảy" dùng `l = max(l, last[x] + 1)`.
- [ ] Cập nhật `best` theo **tổng**, ở **mọi** `r`.
- [ ] Duy trì tổng `O(1)` mỗi bước (biến `s` hoặc prefix), không `sum()` lại.
- [ ] Thử mảng 1 phần tử, toàn trùng, toàn khác.